# Lab 1 (simplified) · Part 3 — The network as a small service
### EUROCONTROL AI Lab · Exercise 1S-3

Other systems can only use a model if they can **ask it**. Here the network from Part 1 runs as a small **web service** on this computer
(`127.0.0.1`; in Google Colab, on the Colab machine). The whole service is the short file **`serve_simple.py`** — open it: about 60 lines, every one commented.

```
 this notebook ── {"day": "2025-12-01", "last_7_days": [...]} ──►  serve_simple.py  ──►  {"forecast": 541.3}
```

| | |
|---|---|
| ⏱ **Time** | about 30 minutes |
| 🎓 **Prerequisites** | Part 1 (it saves the model). A ready-trained model is included, so this part also works on its own. |

> **How to use this notebook**
> - Run the cells **one at a time, from top to bottom**: click a cell and press **Shift + Enter**.
> - Every line of code has a comment (the text after `#`) explaining what it does. You do not need to write any code.
> - After each plot there is a short **📖 Reading** that explains what the plot tells us.
> - If something breaks, restart and run everything again: *Kernel → Restart Kernel and Run All Cells* (JupyterLab) or *Runtime → Restart session and run all* (Colab).
> - This notebook is yours to keep: you can re-run it later and change the numbers to experiment.

> ☁️ **Running in Google Colab?** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dblue-tech/ai-lab/blob/main/lab1_simplified/1S-3_serve.ipynb)  
> Run the next cell **first**: it copies the lab files (data, code, trained models) from GitHub into Colab, so that the rest of the notebook works exactly as on a PC. Wait for *"Colab ready"* (about half a minute).  
> To keep your own copy with your results: *File → Save a copy in Drive*.  
> **On a lab PC** the next cell does nothing: just run it and continue.

In [1]:
# ---- Only for Google Colab: get the lab files. On a lab PC this cell does nothing. ----
import os                                          # os: folders and files
import sys                                         # sys: information about the running Python
import subprocess                                  # subprocess: runs other programs (git, pip)

IN_COLAB = "google.colab" in sys.modules           # True when this notebook runs in Google Colab
if IN_COLAB:                                       # in Colab only:
    if not os.path.exists("/content/ai-lab"):      # if the lab files are not here yet...
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dblue-tech/ai-lab.git", "/content/ai-lab"], check=True)   # ...copy them from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "fastapi", "uvicorn"], check=True)   # libraries for the service, not pre-installed in Colab
    os.chdir("/content/ai-lab/lab1_simplified")    # work inside this notebook's folder, as on a PC
    print("Colab ready — working in", os.getcwd())  # confirm
else:                                              # on a lab PC:
    print("Running on this computer: nothing to do")   # the files are already here

Running on this computer: nothing to do


---
## 1 · Start the service

In [2]:
import sys                                        # sys: the path of the Python program
import time                                       # time: waiting
import subprocess                                 # subprocess: starts the service in the background
import requests                                   # requests: sends messages to the service

PORT = 8000                                       # the "door number" of the service
URL = "http://127.0.0.1:" + str(PORT)             # its address on this computer
IN_COLAB = "google.colab" in sys.modules          # True when running in Google Colab

command = [sys.executable, "-m", "uvicorn", "serve_simple:app", "--host", "127.0.0.1", "--port", str(PORT)]   # the command
server = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)   # start the service
for attempt in range(60):                         # wait up to 30 seconds...
    try:                                          # ...trying to contact it
        if requests.get(URL + "/health", timeout=1).ok:   # it answers:
            break                                 # stop waiting
    except requests.ConnectionError:              # nobody answers yet...
        time.sleep(0.5)                           # ...wait half a second
print("Service answers:", requests.get(URL + "/health").json())   # final check

Service answers: {'status': 'ok', 'hidden_neurons': 16}


The service describes itself on its **documentation page**. Run the next cell and open the link in a new browser tab.

In [3]:
if IN_COLAB:                                       # in Colab the service runs on Google's machine, not on your PC...
    from google.colab import output                # ...so Colab gives us a special link to reach it
    output.serve_kernel_port_as_window(PORT, path="/docs", anchor_text="Service documentation")   # show the link
else:                                              # on a lab PC:
    print("Service documentation:", URL + "/docs") # the address on this computer

Service documentation: http://127.0.0.1:8000/docs


---
## 2 · Ask for a forecast
We ask for **Monday 1 December 2025**, sending the real traffic of the 7 days before, 24–30 November (oldest first). The true traffic that day was **532** flights.

In [4]:
request = {"day": "2025-12-01", "last_7_days": [540, 472, 262, 566, 530, 433, 481]}   # what we send (262 = strike day)
answer = requests.post(URL + "/forecast", json=request)          # send it to the /forecast address
print("Status:", answer.status_code)                             # 200 = OK
print("Answer:", answer.json())                                  # the forecast

Status: 200
Answer: {'day': '2025-12-01', 'forecast': 505.6}


📖 **Reading:** the service answers in a few milliseconds. The forecast is a bit low (the true traffic was 532): the strike day of 26 November (262 flights) is among the 7 days and pulls it down. Any planning tool able to send this small message could now use the network.

---
## 3 · The service refuses nonsense

In [5]:
wrong_requests = {                                               # three wrong requests, on purpose
    "only 6 days": {"day": "2025-12-01", "last_7_days": [540, 472, 262, 566, 530, 433]},
    "negative traffic": {"day": "2025-12-01", "last_7_days": [540, 472, -5, 566, 530, 433, 481]},
    "impossible date": {"day": "2025-13-45", "last_7_days": [540, 472, 262, 566, 530, 433, 481]},
}
for name in wrong_requests:                                      # for each wrong request...
    answer = requests.post(URL + "/forecast", json=wrong_requests[name])   # ...send it...
    print(name, "→ status", answer.status_code, "·", answer.json()["detail"][0]["msg"])   # ...and show the refusal

only 6 days → status 422 · List should have at least 7 items after validation, not 6
negative traffic → status 422 · Input should be greater than or equal to 0
impossible date → status 422 · Input should be a valid date or datetime, month value is outside expected range of 1-12


📖 **Reading:** status **422** means *"your data breaks the rules"*: impossible requests never reach the network.

---
## 4 · Stop the service

In [6]:
server.terminate()                                # ask the service to stop
server.wait(timeout=10)                           # wait until it has stopped
print("Service stopped")                          # confirm

Service stopped


---
## 5 · Questions for discussion
1. The service needs the traffic of the last 7 days. Who would provide these numbers in real operations, and when?
2. What else should the service check before answering?